# Iris Data Analysis

This notebook provides a thorough analysis of the Iris dataset, covering statistical exploration and machine learning model training.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.cluster import KMeans

# Set plot style
sns.set(style="whitegrid")
import warnings
warnings.filterwarnings('ignore')

## Data Loading

Load the dataset and perform initial inspection.

In [ ]:
df = pd.read_csv('iris.csv')

print("First 5 rows:")
display(df.head())

print("\nData Info:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

## Basic Statistics

Examine the central tendency and dispersion of the features.

In [ ]:
print("Statistical Summary:")
display(df.describe())

print("\nClass Distribution:")
print(df['variety'].value_counts())

## Univariate Analysis

Analyze each feature individually using distributions and boxplots.

In [ ]:
features = ['sepal.length', 'sepal.width', 'petal.length', 'petal.width']

# Distribution plots
plt.figure(figsize=(15, 10))
for i, feature in enumerate(features):
    plt.subplot(2, 2, i+1)
    sns.histplot(data=df, x=feature, kde=True, hue='variety', palette='muted')
    plt.title(f'Distribution of {feature}')
plt.tight_layout()
plt.show()

# Boxplots for outlier detection
plt.figure(figsize=(15, 10))
for i, feature in enumerate(features):
    plt.subplot(2, 2, i+1)
    sns.boxplot(data=df, y=feature, x='variety', hue='variety', palette='Set2', legend=False)
    plt.title(f'Boxplot of {feature} by Variety')
plt.tight_layout()
plt.show()

## Bivariate Analysis

Analyze the relationships between pairs of features.

In [ ]:
# Pairplot to see all relationships
sns.pairplot(df, hue='variety', palette='bright', diag_kind='kde')
plt.suptitle('Pairplot of Iris Features', y=1.02)
plt.show()

# Scatter plot for Petal dimensions (highest significance)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='petal.length', y='petal.width', hue='variety', style='variety', palette='deep', s=100)
plt.title('Petal Length vs Petal Width')
plt.show()

## Correlation Analysis

Identify strong correlations between morphological traits.

In [ ]:
corr_matrix = df.drop('variety', axis=1).corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()

## Advanced Analysis: Clustering

Using K-Means to identify natural groupings in the data.

In [ ]:
X_cluster = df.drop('variety', axis=1)
kmeans = KMeans(n_clusters=3, random_state=42)
df['cluster'] = kmeans.fit_predict(X_cluster)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='petal.length', y='petal.width', hue='cluster', palette='viridis', style='variety', s=100)
plt.title('K-Means Clustering vs Actual Variety')
plt.show()

# Drop cluster for ML models
df = df.drop('cluster', axis=1)

## Machine Learning Preparation

Prepare the data for training models.

In [ ]:
# Encoding the target variable
le = LabelEncoder()
df['variety_encoded'] = le.fit_transform(df['variety'])

X = df.drop(['variety', 'variety_encoded'], axis=1)
y = df['variety_encoded']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Scaling features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

## Model Training and Evaluation

Train various classifiers and compare their performance.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100)
}

for name, model in models.items():
    # Train
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    # Evaluate
    acc = accuracy_score(y_test, y_pred)
    print(f"\n{'='*20}")
    print(f"Model: {name}")
    print(f"Accuracy: {acc:.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

## Conclusion

The analysis confirms that petal dimensions are the most significant features for classifying Iris species. All models achieved high accuracy, with Setosa being perfectly separable, while Versicolor and Virginica show slight overlap.